# Faster-Whisper ASR Prototype

Upload prerecorded WAV, MP3, M4A, or WebM product-query clips, transcribe them with `faster-whisper`, and evaluate product terms and word error rate.


## 1. Install dependencies


In [2]:
!pip install -q faster-whisper jiwer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 96.9 MB/s eta 0:00:00


## 2. Check runtime and load the model

For Colab, select **Runtime → Change runtime type → T4 GPU** when available. The code falls back to CPU automatically.


In [3]:
import torch
from faster_whisper import WhisperModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"
MODEL_SIZE = "small.en"

print(f"Device: {DEVICE}")
print(f"Compute type: {COMPUTE_TYPE}")
print(f"Model: {MODEL_SIZE}")

model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
)


Device: cuda
Compute type: float16
Model: small.en


## 3. Upload prerecorded audio files


In [4]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

SUPPORTED_EXTENSIONS = {".wav", ".mp3", ".m4a", ".webm", ".flac", ".ogg"}
audio_files = [
    filename
    for filename in uploaded
    if Path(filename).suffix.lower() in SUPPORTED_EXTENSIONS
]

if not audio_files:
    raise ValueError("No supported audio files were uploaded.")

print(f"Uploaded {len(audio_files)} audio file(s):")
for filename in audio_files:
    print("-", filename)


Saving query10.wav to query10.wav
Saving query09.wav to query09.wav
Saving query08.wav to query08.wav
Saving query07.wav to query07.wav
Saving query06.wav to query06.wav
Saving query05.wav to query05.wav
Saving query04.wav to query04.wav
Saving query03.wav to query03.wav
Saving query02.wav to query02.wav
Saving query01.wav to query01.wav
Uploaded 10 audio file(s):
- query10.wav
- query09.wav
- query08.wav
- query07.wav
- query06.wav
- query05.wav
- query04.wav
- query03.wav
- query02.wav
- query01.wav


## 4. Define the pipeline-facing transcription function


In [5]:
from pathlib import Path
from typing import Any


def transcribe(audio_file: str | Path) -> dict[str, Any]:
    """Transcribe one short English product-query audio file."""
    audio_path = Path(audio_file)

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    segments, info = model.transcribe(
        str(audio_path),
        language="en",
        beam_size=5,
        vad_filter=True,
        word_timestamps=True,
        condition_on_previous_text=False,
    )

    transcript_parts: list[str] = []
    segment_results: list[dict[str, Any]] = []

    # Iterating over segments performs the transcription.
    for segment in segments:
        segment_text = segment.text.strip()

        if segment_text:
            transcript_parts.append(segment_text)

        words = [
            {
                "word": word.word.strip(),
                "start": word.start,
                "end": word.end,
                "confidence": word.probability,
            }
            for word in (segment.words or [])
        ]

        segment_results.append(
            {
                "start": segment.start,
                "end": segment.end,
                "text": segment_text,
                "words": words,
            }
        )

    return {
        "text": " ".join(transcript_parts).strip(),
        "language": info.language,
        "language_probability": info.language_probability,
        "segments": segment_results,
        "source_file": str(audio_path),
    }


## 5. Transcribe all uploaded files


In [6]:
transcription_results = []

for audio_file in audio_files:
    output = transcribe(audio_file)
    transcription_results.append(output)

    print(f"\n{audio_file}")
    print("-" * len(audio_file))
    print(output["text"])
    print(
        f"Language: {output['language']} "
        f"({output['language_probability']:.1%} confidence)"
    )



query10.wav
-----------
Show products with 4.5 stars or higher.
Language: en (100.0% confidence)

query09.wav
-----------
Best cleaner for granite countertops.
Language: en (100.0% confidence)

query08.wav
-----------
Highest rated stainless steel cleaner.
Language: en (100.0% confidence)

query07.wav
-----------
Find Freakin's Free Cleaner.
Language: en (100.0% confidence)

query06.wav
-----------
Compare method and Mrs. Meyers.
Language: en (100.0% confidence)

query05.wav
-----------
Show products under $12.99.
Language: en (100.0% confidence)

query04.wav
-----------
Find a plant-based kitchen cleaner
Language: en (100.0% confidence)

query03.wav
-----------
Show me 7th generation products.
Language: en (100.0% confidence)

query02.wav
-----------
Compare women with therapy clean.
Language: en (100.0% confidence)

query01.wav
-----------
Recommend an eco-friendly stainless steel cleaner under $15.
Language: en (100.0% confidence)


## 6. Review transcripts and timestamps


In [7]:
import pandas as pd

transcripts_df = pd.DataFrame(
    {
        "file": result["source_file"],
        "transcript": result["text"],
        "language": result["language"],
        "language_confidence": result["language_probability"],
    }
    for result in transcription_results
)

transcripts_df


,file,transcript,language,language_confidence
0,query10.wav,Show products with 4.5 stars or higher.,en,1
1,query09.wav,Best cleaner for granite countertops.,en,1
2,query08.wav,Highest rated stainless steel cleaner.,en,1
3,query07.wav,Find Freakin's Free Cleaner.,en,1
4,query06.wav,Compare method and Mrs. Meyers.,en,1
5,query05.wav,Show products under $12.99.,en,1
6,query04.wav,Find a plant-based kitchen cleaner,en,1
7,query03.wav,Show me 7th generation products.,en,1
8,query02.wav,Compare women with therapy clean.,en,1
9,query01.wav,Recommend an eco-friendly stainless steel clea...,en,1


In [8]:
# Inspect word timestamps and confidence for one uploaded file.
selected_result = transcription_results[0]

print("File:", selected_result["source_file"])
print("Transcript:", selected_result["text"])

for segment in selected_result["segments"]:
    print(f"\n[{segment['start']:.2f}s–{segment['end']:.2f}s] {segment['text']}")
    for word in segment["words"]:
        print(
            f"  {word['word']:<20} "
            f"{word['start']:>6.2f}s–{word['end']:>6.2f}s "
            f"confidence={word['confidence']:.3f}"
        )


File: query10.wav
Transcript: Show products with 4.5 stars or higher.

[0.00s–3.26s] Show products with 4.5 stars or higher.
  Show                   0.00s–  0.60s confidence=0.835
  products               0.60s–  1.02s confidence=0.875
  with                   1.02s–  1.24s confidence=0.994
  4                      1.24s–  1.60s confidence=0.989
  .5                     1.60s–  2.38s confidence=0.987
  stars                  2.38s–  2.70s confidence=0.937
  or                     2.70s–  3.00s confidence=0.984
  higher.                3.00s–  3.26s confidence=1.000


## 7. Add reference transcripts


In [9]:
reference_queries = {
    "query01.wav": (
        "Recommend an eco-friendly stainless-steel cleaner "
        "under fifteen dollars."
    ),
    "query02.wav": (
        "Compare Weiman with Therapy Clean."
    ),
    "query03.wav": (
        "Show me Seventh Generation products."
    ),
    "query04.wav": (
        "Find a plant-based kitchen cleaner."
    ),
    "query05.wav": (
        "Show products under twelve ninety-nine."
    ),
    "query06.wav": (
        "Compare Method and Mrs. Meyer's."
    ),
    "query07.wav": (
        "Find fragrance-free cleaner."
    ),
    "query08.wav": (
        "Highest rated stainless steel cleaner."
    ),
    "query09.wav": (
        "Best cleaner for granite countertops."
    ),
    "query10.wav": (
        "Show products with four-point-five stars or higher."
    )
}

print("Reference transcripts added:", len(reference_queries))


Reference transcripts added: 10


## 8. Evaluate word error rate


In [19]:
from jiwer import wer

evaluation_results = []

for audio_file in audio_files:

    if audio_file not in reference_queries:
        continue

    result = transcribe(audio_file)

    reference = reference_queries[audio_file]
    transcript = result["text"]

    evaluation_results.append({
        "file": audio_file,
        "reference": reference,
        "transcript": transcript,
        "wer": wer(
            normalize_text(reference),
            normalize_text(transcript)
        )
    })

evaluation_df = pd.DataFrame(evaluation_results)
display(evaluation_df)

,file,reference,transcript,wer
0,query10.wav,Show products with four-point-five stars or hi...,Show products with 4.5 stars or higher.,0.00
1,query09.wav,Best cleaner for granite countertops.,Best cleaner for granite countertops.,0.00
2,query08.wav,Highest rated stainless steel cleaner.,Highest rated stainless steel cleaner.,0.00
3,query07.wav,Find fragrance-free cleaner.,Find Freakin's Free Cleaner.,0.25
4,query06.wav,Compare Method and Mrs. Meyer's.,Compare method and Mrs. Meyers.,0.00
5,query05.wav,Show products under twelve ninety-nine.,Show products under $12.99.,0.00
6,query04.wav,Find a plant-based kitchen cleaner.,Find a plant-based kitchen cleaner,0.00
7,query03.wav,Show me Seventh Generation products.,Show me 7th generation products.,0.00
8,query02.wav,Compare Weiman with Therapy Clean.,Compare women with therapy clean.,0.20
9,query01.wav,Recommend an eco-friendly stainless-steel clea...,Recommend an eco-friendly stainless steel clea...,0.00


## 9. Check critical product terms


In [20]:
import re
from pathlib import Path
import pandas as pd


def normalize_text(text: str) -> str:
    """Normalize text before comparison."""

    text = text.lower().strip()

    # Normalize punctuation first
    text = text.replace("-", " ")
    text = text.replace("–", " ")
    text = text.replace("—", " ")

    # Normalize common ASR formatting differences
    replacements = {
        "$15": "fifteen dollars",
        "$12.99": "twelve ninety nine",
        "$12 99": "twelve ninety nine",
        "4.5": "four point five",
        "4 5": "four point five",
        "7th": "seventh",
        "stainless-steel": "stainless steel",
        "fragrance-free": "fragrance free",
        "plant-based": "plant based",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    # Remove apostrophes (Mrs. Meyer's -> mrs meyers)
    text = re.sub(r"[’']", "", text)

    # Remove remaining punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def check_terms(transcript: str, expected_terms: list[str]) -> dict[str, bool]:
    """Check whether expected product terms appear in the transcript."""

    normalized_transcript = normalize_text(transcript)

    return {
        term: normalize_text(term) in normalized_transcript
        for term in expected_terms
    }


# Expected keywords for each audio file.
expected_terms_by_file = {
    "query01.wav": [
        "eco-friendly",
        "stainless-steel",
        "fifteen dollars",
    ],
    "query02.wav": [
        "weiman",
        "therapy clean",
    ],
    "query03.wav": [
        "seventh generation",
    ],
    "query04.wav": [
        "plant-based",
        "kitchen cleaner",
    ],
    "query05.wav": [
        "twelve ninety-nine",
    ],
    "query06.wav": [
        "method",
        "mrs. meyer's",
    ],
    "query07.wav": [
        "fragrance-free",
        "cleaner",
    ],
    "query08.wav": [
        "highest rated",
        "stainless steel cleaner",
    ],
    "query09.wav": [
        "granite countertops",
    ],
    "query10.wav": [
        "four-point-five stars",
        "higher",
    ],
}

term_records = []

for result in transcription_results:
    filename = Path(result["source_file"]).name

    expected_terms = expected_terms_by_file.get(filename)

    if not expected_terms:
        print(f"Skipping {filename}: no expected terms defined.")
        continue

    matches = check_terms(result["text"], expected_terms)

    for term, matched in matches.items():
        term_records.append(
            {
                "file": filename,
                "expected_term": term,
                "recognized": matched,
            }
        )

term_accuracy_df = pd.DataFrame(term_records)

if term_accuracy_df.empty:
    print("No term evaluations were performed.")
else:
    display(term_accuracy_df)
    print(f"\nOverall term accuracy: {term_accuracy_df['recognized'].mean():.1%}")

,file,expected_term,recognized
0,query10.wav,four-point-five stars,True
1,query10.wav,higher,True
2,query09.wav,granite countertops,True
3,query08.wav,highest rated,True
4,query08.wav,stainless steel cleaner,True
5,query07.wav,fragrance-free,False
6,query07.wav,cleaner,True
7,query06.wav,method,True
8,query06.wav,mrs. meyer's,True
9,query05.wav,twelve ninety-nine,True



Overall term accuracy: 88.9%


In [24]:
if not evaluation_df.empty:
    print(f"Average WER: {evaluation_df['wer'].mean():.3f}")

if not term_accuracy_df.empty:
    term_accuracy = term_accuracy_df["recognized"].mean()
    print(f"Critical-term accuracy: {term_accuracy:.1%}")

Average WER: 0.045
Critical-term accuracy: 88.9%
